# Granger Causality: Uncovers potential causal relationships between genes over time. 

Pre-requisites:
- Trained MIOFlow trajectories

In this notebook:
- Perform Granger causality testing:
    - Between each pair of genes (or a subset of input/output gene groups).

    - Using lag-1 autoregressive models.

- Output a p-value matrix:

    - Rows: Response genes

    - Columns: Predictor genes

    - Values: Significance of Granger-causal influence

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
from statsmodels.tsa.stattools import grangercausalitytests
lag_order = 1 # since we aggregated the data in to 9 bins we only need 1 lag
maxlag = (
    lag_order,  # becuase we got this value before. We are not suppose to add 1 to it
)
test = "ssr_chi2test"
import scanpy as sc
from joblib import Parallel, delayed

In [3]:
def grangers_causation_matrix(
    data, in_variables, out_variables, test="ssr_chi2test", n_jobs=1, warn=False
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """

    def get_pval(dd):
        if warn:
            test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=True)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=FutureWarning)
                test_result = grangercausalitytests(dd, maxlag=maxlag, verbose=False)
                # according to the documentation https://www.statsmodels.org/dev/generated/statsmodels.tsa.stattools.grangercausalitytests.html,
                # the dd has 2 columns, second causes the first.

        p_values = [test_result[i][0][test][1] for i in maxlag] # test_result[i][1] is the unrestricted model, test_result[i][1][0] is the restricted model
        coefs = [test_result[i][1][1].params[1] for i in maxlag] # x1, x2, const

        arg_min_p_value = np.argmin(p_values)
        min_p_value = p_values[arg_min_p_value]
        min_coef = coefs[arg_min_p_value]
        return (min_p_value, min_coef)

    out = Parallel(n_jobs=n_jobs)(
        delayed(get_pval)(data[[c, r]]) # this means r causes c, so r is be in and c is out
        for c in tqdm(out_variables, desc="Processing columns")  # Outer loop progress bar
        for r in in_variables  # Inner loop without progress bar
    )
    out_p = [p for (p,c) in out]
    out_c = [c for (p,c) in out]
    df_p = pd.DataFrame(
        np.array(out_p).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T # used the correct reshaping, and then transposed the matrix so the x and y are semantically correct (x causes y).
    df_c = pd.DataFrame(
        np.array(out_c).reshape((len(out_variables), len(in_variables))), # should be reshaped to len(out_variables), len(in_variables) according to the for loop.
        columns=in_variables,
        index=out_variables,
    ).T
    df_p.index = [var + "_x" for var in in_variables]
    df_p.columns = [var + "_y" for var in out_variables]
    df_c.index = [var + "_x" for var in in_variables]
    df_c.columns = [var + "_y" for var in out_variables]
    return df_p, df_c

def do_granger(trajs, in_genes, out_genes, n_jobs=1, warn=False):
    # in causes out
    trajs = trajs.T[::10]
    trajs = trajs - trajs.shift(1)
    trajs = trajs.dropna()
    out_traj_p, out_traj_c = grangers_causation_matrix(
        trajs, in_variables=in_genes, out_variables=out_genes, n_jobs=n_jobs, warn=warn
    )
    return out_traj_p, out_traj_c

Load the gene expression trajectories generated by MIOFlow along with the original dataset. For simplicity, we focus on the top 2,000 most highly variable genes.

In [4]:
trajectories = np.load('../../results/scRNAseq/trajectories_gene_space.npy', allow_pickle=True)
adata = sc.read('../../data/processed/adata_mioflow.h5ad')
genes = adata.var_names.to_list()
sc.pp.highly_variable_genes(adata, n_top_genes=250)
hv_mask = adata.var['highly_variable']
genes_hvg = hv_mask[hv_mask].index.tolist()
col_genes = np.array(genes_hvg)

Intersect the selected genes with the list of human transcription factors downloaded from humantfs.ccbr.utoronto.ca to identify the input genes that are transcription factors.

In [5]:
db_extract = pd.read_csv('DatabaseExtract_v_1.01.csv')
db_ensembl_ids = set(db_extract['Ensembl ID'].astype(str))
genes_ensembl_ids = set([g.split('(')[-1].replace(')', '').strip() for g in genes_hvg])
in_genes_ensembl = genes_ensembl_ids & db_ensembl_ids
in_genes = [g for g in genes_hvg if g.split('(')[-1].replace(')', '').strip() in in_genes_ensembl]

In [6]:
trajectories_hvg = trajectories[:, :, hv_mask.values]
avg_traj = trajectories_hvg.mean(axis=1)
trajectories_df = pd.DataFrame(avg_traj, columns=col_genes)
out_traj_p, out_traj_c = do_granger(trajectories_df.T, in_genes, genes_hvg, n_jobs=1, warn=False)

Processing columns: 100%|██████████| 250/250 [00:04<00:00, 57.55it/s]


Output of Granger Causality:

1. out_traj_p: p-value (smaller the better)
2. out_traj_c: coefficient (the sign indicates if it is up or down regulation)

In [7]:
out_traj_p

,AC007325.4 (ENSG00000278817)_y,ACTA1 (ENSG00000143632)_y,AFP (ENSG00000081051)_y,AGT (ENSG00000135744)_y,AHSG (ENSG00000145192)_y,ALDH1A1 (ENSG00000165092)_y,ALX1 (ENSG00000180318)_y,ANKRD37 (ENSG00000186352)_y,APOM (ENSG00000204444)_y,AREG (ENSG00000109321)_y,...,TSPAN8 (ENSG00000127324)_y,TWIST2 (ENSG00000233608)_y,TYRP1 (ENSG00000107165)_y,VTN (ENSG00000109072)_y,VWF (ENSG00000110799)_y,WFDC1 (ENSG00000103175)_y,WIF1 (ENSG00000156076)_y,XAGE2 (ENSG00000155622)_y,ZFP36 (ENSG00000128016)_y,ZNF592 (ENSG00000166716)_y
AGT (ENSG00000135744)_x,6.883337e-02,6.358144e-01,3.081689e-01,1.000000e+00,7.443595e-01,1.395107e-09,1.384920e-28,7.568721e-08,5.455328e-11,2.230475e-102,...,2.476855e-19,5.411588e-01,8.349227e-01,2.460706e-01,1.739364e-40,1.363880e-11,5.817504e-02,1.450896e-06,0.977715,1.689323e-03
ALX1 (ENSG00000180318)_x,8.503231e-01,1.190601e-02,1.712670e-05,2.799117e-01,1.788974e-01,5.063907e-05,1.000000e+00,3.191003e-05,1.464008e-01,2.001472e-01,...,8.975927e-04,3.132397e-01,8.441650e-03,2.983004e-01,5.975362e-09,3.289290e-10,1.600957e-01,4.191520e-02,0.069340,2.361777e-07
ASCL1 (ENSG00000139352)_x,8.550777e-01,3.453263e-03,1.153450e-05,2.326575e-01,1.464324e-01,8.248693e-06,1.727246e-01,9.544506e-08,1.077443e-01,1.573090e-01,...,5.834283e-05,4.069218e-01,1.043375e-02,3.030177e-01,1.865065e-11,1.121996e-14,2.195732e-01,2.734113e-02,0.078926,2.647520e-06
ATOH1 (ENSG00000172238)_x,8.796005e-01,6.814380e-01,1.770648e-01,9.306099e-01,7.666297e-01,3.108585e-02,3.263485e-01,5.088257e-01,8.047746e-01,9.066428e-01,...,9.757091e-01,3.274393e-03,2.082064e-107,1.060029e-02,1.531380e-02,2.920156e-02,2.177731e-01,5.031475e-01,0.850702,2.927668e-04
CDX4 (ENSG00000131264)_x,1.543682e-02,4.020980e-01,5.036789e-01,2.857232e-02,1.542458e-04,2.504393e-02,3.747203e-26,3.302795e-03,1.494420e-01,8.754046e-02,...,1.017938e-16,6.477538e-01,2.199635e-01,5.810551e-01,1.761249e-04,3.940101e-07,2.335033e-02,4.107388e-03,0.423155,3.166915e-11
CYP1B1 (ENSG00000138061)_x,2.487113e-02,6.259232e-01,9.628221e-01,8.016469e-02,9.828372e-03,1.263647e-04,1.578352e-20,6.331174e-04,4.247765e-01,3.045680e-01,...,8.651421e-17,5.667910e-01,3.362545e-01,6.462219e-01,5.125958e-27,2.761060e-08,2.393461e-02,2.033722e-01,0.802811,2.650568e-10
DLX5 (ENSG00000105880)_x,1.930240e-02,3.049412e-01,1.309570e-01,1.229776e-01,3.643176e-02,5.122798e-01,2.442937e-10,1.954535e-02,2.049959e-01,1.963213e-01,...,2.029558e-04,4.593974e-01,1.368253e-02,9.107070e-01,6.620783e-01,1.027699e-02,2.700348e-02,4.219970e-02,0.541525,1.950706e-25
EDN1 (ENSG00000078401)_x,2.648872e-04,1.512454e-01,4.699192e-01,5.007251e-10,1.901380e-90,1.130543e-01,9.003792e-26,3.103795e-02,2.283449e-05,1.039340e-07,...,1.221883e-06,8.042084e-01,2.284931e-01,1.783142e-01,4.723044e-05,6.052621e-48,1.231654e-03,1.974558e-77,0.289651,8.197062e-18
FAM200B (ENSG00000237765)_x,4.077764e-14,7.260752e-02,1.751824e-01,8.897222e-01,8.905599e-01,2.921346e-03,1.889732e-07,3.499787e-03,6.786061e-01,8.455626e-01,...,2.985765e-03,1.776444e-02,3.122507e-01,1.044536e-02,2.806979e-07,3.312029e-23,1.386098e-09,3.570522e-01,0.611102,1.171298e-02
GSX2 (ENSG00000180613)_x,4.301555e-02,9.695100e-01,2.208125e-01,7.769451e-01,9.725143e-01,6.049778e-02,7.959618e-01,8.579783e-01,9.912062e-01,8.781530e-01,...,3.799491e-01,1.936681e-04,2.537084e-02,1.274631e-03,6.847112e-03,5.143549e-04,2.753251e-04,6.437356e-01,0.997278,1.388935e-02


Save the p-values and coefficients to use for RITINI

In [8]:
out_traj_p.to_csv('../../data/processed/out_traj_p_250.csv')
out_traj_c.to_csv('../../data/processed/out_traj_c_250.csv')